In [ ]:
import time
import psutil
import os
from ptflops import get_model_complexity_info

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import random
from torch.optim import LBFGS, Adam
from tqdm import tqdm
import copy
from model_components.models import PINNs
from model_components.util import *

def get_memory_usage():
    """Get current memory usage"""
    process = psutil.Process(os.getpid())
    cpu_mem = process.memory_info().rss / 1024**2  # MB
    gpu_mem = torch.cuda.memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
    return cpu_mem, gpu_mem


def count_model_flops(model, device):
    try:
        # Create wrapper model that takes single input
        class ModelWrapper(nn.Module):
            def __init__(self, original_model):
                super().__init__()
                self.model = original_model
                
            def forward(self, combined_input):
                # Split combined input back into x and t
                batch_size, seq_len, features = combined_input.shape
                x = combined_input[:, :, :1]  # First feature is x
                t = combined_input[:, :, 1:2]  # Second feature is t
                return self.model(x, t)
        
        wrapped_model = ModelWrapper(model)
        
        macs, params = get_model_complexity_info(
            wrapped_model, 
            (5, 2),  # sequence length, features (x and t)
            print_per_layer_stat=False,
            verbose=False
        )
        
        # Debug: Print what we actually get
        print(f"Raw MACs output: {macs}")
        print(f"MACs split: {macs.split(' ')}")
        
        # More robust parsing
        mac_parts = macs.split(' ')
        mac_value = float(mac_parts[0])
        mac_unit = mac_parts[1] if len(mac_parts) > 1 else "Mac"
        
        print(f"Parsed MAC value: {mac_value}")
        print(f"MAC unit: {mac_unit}")
        
        # Convert to FLOPs based on unit
        if "MMac" in mac_unit:
            # Million MACs
            flops = mac_value * 1e6 * 2  # 2 FLOPs per MAC
        elif "GMac" in mac_unit:
            # Billion MACs  
            flops = mac_value * 1e9 * 2
        elif "KMac" in mac_unit:
            # Thousand MACs
            flops = mac_value * 1e3 * 2
        else:
            # Default assume base MACs
            flops = mac_value * 2
            
        print(f"Calculated FLOPs: {flops:,.0f} ({flops/1e9:.3f} GFLOPs)")
        
        return flops, params
        
    except Exception as e:
        print(f"FLOP counting with ptflops failed: {e}")
        return None, None


# Set seeds for reproducibility
# Set seeds for reproducibility
seed = 0
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

step_size = 1e-4
# Train PINNsformer
res, b_left, b_right, b_upper, b_lower = get_data([0,2*np.pi], [0,1], 51, 51)
res_test, _, _, _, _ = get_data([0,2*np.pi], [0,1], 101, 101)

res, b_left, b_right, b_upper, b_lower = get_data([0,2*np.pi], [0,1], 101, 101)
res_test, _, _, _, _ = get_data([0,2*np.pi], [0,1], 101, 101)

res = torch.tensor(res, dtype=torch.float32, requires_grad=True).to(device)
b_left = torch.tensor(b_left, dtype=torch.float32, requires_grad=True).to(device)
b_right = torch.tensor(b_right, dtype=torch.float32, requires_grad=True).to(device)
b_upper = torch.tensor(b_upper, dtype=torch.float32, requires_grad=True).to(device)
b_lower = torch.tensor(b_lower, dtype=torch.float32, requires_grad=True).to(device)

x_res, t_res = res[:,0:1], res[:,1:2]
x_left, t_left = b_left[:,0:1], b_left[:,1:2]
x_right, t_right = b_right[:,0:1], b_right[:,1:2]
x_upper, t_upper = b_upper[:,0:1], b_upper[:,1:2]
x_lower, t_lower = b_lower[:,0:1], b_lower[:,1:2]

def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)

# Note: d_model (here 32) should be even for the Fourier features mapping.
model = PINNs(in_dim=2, hidden_dim=512, out_dim=1, num_layer=4).to(device)

model.apply(init_weights)
optim = LBFGS(model.parameters(), line_search_fn='strong_wolfe')
# optim = Adam(model.parameters(), lr = 1e-4)

print(model)
print(get_n_params(model))

n_params = get_n_params(model)



kernel_size = 300

D1 = kernel_size
D2 = len(x_left)
D3 = len(x_lower)

def compute_ntk(J1, J2):
    Ker = torch.matmul(J1, torch.transpose(J2, 0, 1))
    return Ker


PINNs(
  (linear): Sequential(
    (0): Linear(in_features=2, out_features=512, bias=True)
    (1): Tanh()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): Tanh()
    (4): Linear(in_features=512, out_features=512, bias=True)
    (5): Tanh()
    (6): Linear(in_features=512, out_features=1, bias=True)
  )
)
527361


In [ ]:
# Count FLOPs
flops, params = count_model_flops(model, device)
if flops:
    print(f"FLOPs per forward pass: {flops:.6f} ({flops/1e9:.3f} GFLOPs)")

# Initialize timing and memory tracking
step_times = []
memory_snapshots = []
epoch_start_time = time.time()

print("=== STARTING PROFILED TRAINING ===")

loss_track = []
w1, w2, w3 = 1, 1, 1

pi = torch.tensor(np.pi, dtype=torch.float32, requires_grad=False).to(device)

progress_bar = tqdm(range(500))

for i in progress_bar:
    # Record step start time and memory
    step_start_time = time.time()
    cpu_mem_before, gpu_mem_before = get_memory_usage()
    
    if i % 20 == 0:
        J1 = torch.zeros((D1, n_params))
        J2 = torch.zeros((D2, n_params))
        J3 = torch.zeros((D3, n_params))

        batch_ind = np.random.choice(len(x_res), kernel_size, replace=False)
        x_train, t_train = x_res[batch_ind], t_res[batch_ind]

        pred_res = model(x_train, t_train)
        pred_left = model(x_left, t_left)
        pred_upper = model(x_upper, t_upper)
        pred_lower = model(x_lower, t_lower)

        for j in range(len(x_train)):
            model.zero_grad()
            pred_res[j,0].backward(retain_graph=True)
            J1[j, :] = torch.cat([
                p.grad.view(-1) if p.grad is not None else torch.zeros_like(p).view(-1) 
                for p in model.parameters()
                ])


        for j in range(len(x_left)):
            model.zero_grad()
            pred_left[j,0].backward(retain_graph=True)
            J2[j, :] = torch.cat([
                p.grad.view(-1) if p.grad is not None else torch.zeros_like(p).view(-1) 
                for p in model.parameters()
                ])

        for j in range(len(x_lower)):
            model.zero_grad()
            pred_lower[j,0].backward(retain_graph=True)
            pred_upper[j,0].backward(retain_graph=True)
            J3[j, :] = torch.cat([
                p.grad.view(-1) if p.grad is not None else torch.zeros_like(p).view(-1) 
                for p in model.parameters()
                ])

        K1 = torch.trace(compute_ntk(J1, J1))
        K2 = torch.trace(compute_ntk(J2, J2))
        K3 = torch.trace(compute_ntk(J3, J3))
        
        K = K1+K2+K3

        w1 = K.item() / K1.item()
        w2 = K.item() / K2.item()
        w3 = K.item() / K3.item()
    
    def closure():
        pred_res = model(x_res, t_res)
        pred_left = model(x_left, t_left)
        pred_right = model(x_right, t_right)
        pred_upper = model(x_upper, t_upper)
        pred_lower = model(x_lower, t_lower)

        u_x = torch.autograd.grad(pred_res, x_res, grad_outputs=torch.ones_like(pred_res), retain_graph=True, create_graph=True)[0]
        u_t = torch.autograd.grad(pred_res, t_res, grad_outputs=torch.ones_like(pred_res), retain_graph=True, create_graph=True)[0]

        loss_res = torch.mean((u_t - 5 * pred_res * (1-pred_res)) ** 2)
        loss_bc = torch.mean((pred_upper - pred_lower) ** 2)
        loss_ic = torch.mean((pred_left[:,0] - torch.exp(- (x_left[:,0] - torch.pi)**2 / (2*(torch.pi/4)**2))) ** 2)

        loss_track.append([loss_res.item(), loss_bc.item(), loss_ic.item()])

        loss = w1 * loss_res + w3 * loss_bc + w2* loss_ic
        optim.zero_grad()
        loss.backward()
        return loss

    optim.step(closure)
            
    last_loss = loss_track[-1]
    progress_bar.set_postfix({
        "res": f"{last_loss[0]:.6f}",
        "ic": f"{last_loss[1]:.6f}",
        "bc": f"{last_loss[2]:.6f}",
    })
    
    # Record timing and memory after step
    step_time = time.time() - step_start_time
    cpu_mem_after, gpu_mem_after = get_memory_usage()
    
    step_times.append(step_time)
    memory_snapshots.append({
        'step': i,
        'cpu_before': cpu_mem_before,
        'cpu_after': cpu_mem_after,
        'gpu_before': gpu_mem_before,
        'gpu_after': gpu_mem_after,
        'gpu_peak': torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
    })
    
    # Update progress bar with timing info
    if len(step_times) > 0:
        last_loss = loss_track[-1]
        progress_bar.set_postfix({
            "res": f"{last_loss[0]:.6f}",
            "ic": f"{last_loss[1]:.6f}", 
            "bc": f"{last_loss[2]:.6f}",
            "step_time": f"{step_time:.3f}s",
            "gpu_mem": f"{gpu_mem_after:.0f}MB"
        })

total_training_time = time.time() - epoch_start_time

# Print profiling summary
print("\n=== PERFORMANCE SUMMARY ===")
print(f"Total training time: {total_training_time:.2f} seconds")
print(f"Average step time: {np.mean(step_times):.3f} seconds")
print(f"Steps per second: {len(step_times)/total_training_time:.2f}")
print(f"Peak GPU memory: {max([m['gpu_peak'] for m in memory_snapshots]):.1f} MB")
print(f"Final GPU memory: {memory_snapshots[-1]['gpu_after']:.1f} MB")
print(f"Average GPU memory during training: {np.mean([m['gpu_after'] for m in memory_snapshots]):.1f} MB")

avg_time_per_100_steps = np.mean([np.mean(step_times[i:i+100]) * 100 
                                   for i in range(0, len(step_times)-100, 100)])
print(f"Time per 100 steps (epoch equivalent): {avg_time_per_100_steps:.2f} seconds")

Raw MACs output: 2.64 MMac
MACs split: ['2.64', 'MMac']
Parsed MAC value: 2.64
MAC unit: MMac
Calculated FLOPs: 5,280,000 (0.005 GFLOPs)
FLOPs per forward pass: 5280000.000000 (0.005 GFLOPs)
=== STARTING PROFILED TRAINING ===


100%|██████████| 500/500 [03:15<00:00,  2.56it/s, res=0.000037, ic=0.000000, bc=0.000008, step_time=0.106s, gpu_mem=430MB]


=== PERFORMANCE SUMMARY ===
Total training time: 195.62 seconds
Average step time: 0.390 seconds
Steps per second: 2.56
Peak GPU memory: 851.6 MB
Final GPU memory: 430.5 MB
Average GPU memory during training: 428.7 MB
Time per 100 steps (epoch equivalent): 42.87 seconds
Profiling data saved to 'profiling_results.json'


In [4]:
# convert FLOPs to MFLOPs
mflops = flops / 1e6 if flops else None
print(f"FLOPs per forward pass: {flops:.0f} ({mflops:.2f} MFLOPs)")

FLOPs per forward pass: 5280000 (5.28 MFLOPs)


In [3]:
# Add profiling setup after model creation
print("=== MODEL ANALYSIS ===")
print(f"Model parameters: {get_n_params(model):,}")

=== MODEL ANALYSIS ===
Model parameters: 527,361
